<a href="https://colab.research.google.com/github/santhoshi-h/datasets/blob/main/chatbot_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

# Load the dataset
file_path = "/content/dataset.csv"
data = pd.read_csv(file_path).dropna(subset=["QUESTION", "ANSWER"])

# Tokenization and preprocessing
class TextTokenizer:
    def __init__(self):
        self.word2index = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
        self.index2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>"}
        self.vocab_size = 3

    def fit_on_texts(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word2index:
                    self.word2index[word] = self.vocab_size
                    self.index2word[self.vocab_size] = word
                    self.vocab_size += 1

    def texts_to_sequences(self, texts):
        return [[self.word2index[word] for word in text.split()] for text in texts]

    def sequences_to_texts(self, sequences):
        return [[self.index2word[idx] for idx in sequence if idx in self.index2word] for sequence in sequences]

# Initialize tokenizers
question_tokenizer = TextTokenizer()
answer_tokenizer = TextTokenizer()

question_tokenizer.fit_on_texts(data["QUESTION"].tolist())
answer_tokenizer.fit_on_texts(data["ANSWER"].tolist())

# Convert texts to sequences
questions_seq = question_tokenizer.texts_to_sequences(data["QUESTION"].tolist())
answers_seq = answer_tokenizer.texts_to_sequences(data["ANSWER"].tolist())

# Add <SOS> and <EOS> tokens to answers
answers_seq = [[1] + seq + [2] for seq in answers_seq]

# Padding
def pad_sequences(sequences, max_len):
    return [seq + [0] * (max_len - len(seq)) if len(seq) < max_len else seq[:max_len] for seq in sequences]

max_question_len = max(len(seq) for seq in questions_seq)
max_answer_len = max(len(seq) for seq in answers_seq)

questions_padded = pad_sequences(questions_seq, max_question_len)
answers_padded = pad_sequences(answers_seq, max_answer_len)

# Split data
train_questions, test_questions, train_answers, test_answers = train_test_split(
    questions_padded, answers_padded, test_size=0.2, random_state=42
)

# Dataset and DataLoader
class ChatbotDataset(Dataset):
    def __init__(self, questions, answers):
        self.questions = torch.tensor(questions, dtype=torch.long)
        self.answers = torch.tensor(answers, dtype=torch.long)

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return self.questions[idx], self.answers[idx]

train_dataset = ChatbotDataset(train_questions, train_answers)
test_dataset = ChatbotDataset(test_questions, test_answers)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Transformer Model
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, query, key, value, mask=None):
        d_k = query.size(-1)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attention = F.softmax(scores, dim=-1)
        return torch.matmul(attention, value), attention

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % n_heads == 0

        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        query = self.query(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        key = self.key(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        value = self.value(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        scaled_attention, _ = ScaledDotProductAttention()(query, key, value, mask)
        scaled_attention = scaled_attention.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)

        return self.out(scaled_attention)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.encoding[:, :seq_len, :].to(x.device)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        self_attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(self_attn_output))

        cross_attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(cross_attn_output))

        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_output))
        return x

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, n_heads, d_ff, num_layers, dropout=0.1):
        super(Transformer, self).__init__()
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model)

        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def make_masks(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(2)
        seq_len = tgt.size(1)
        nopeak_mask = torch.triu(torch.ones((1, seq_len, seq_len), device=tgt.device), diagonal=1).bool()
        tgt_mask = tgt_mask & ~nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.make_masks(src, tgt)

        src = self.src_embedding(src) * math.sqrt(src.size(-1))
        tgt = self.tgt_embedding(tgt) * math.sqrt(tgt.size(-1))

        src = self.positional_encoding(src)
        tgt = self.positional_encoding(tgt)

        for layer in self.encoder_layers:
            src = layer(src, src_mask)

        for layer in self.decoder_layers:
            tgt = layer(tgt, src, src_mask, tgt_mask)

        return self.fc_out(tgt)

# Hyperparameters
d_model = 128
n_heads = 8
d_ff = 512
num_layers = 12
dropout = 0.1

model = Transformer(
    src_vocab_size=question_tokenizer.vocab_size,
    tgt_vocab_size=answer_tokenizer.vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=dropout
)

# Training and Evaluation
import torch.optim as optim
from tqdm import tqdm

# Loss function and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding index
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Adjust learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

# Training loop
def train_model(model, train_loader, optimizer, scheduler, criterion, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for questions, answers in tqdm(train_loader):
            questions, answers = questions.to(device), answers.to(device)

            # Shift answers for teacher forcing
            inputs = answers[:, :-1]
            targets = answers[:, 1:]

            optimizer.zero_grad()
            outputs = model(questions, inputs)
            loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
            optimizer.step()
            total_loss += loss.item()

        # Update the learning rate scheduler
        scheduler.step(total_loss)

        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_loader)}")

def validate_model(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for questions, answers in val_loader:
            questions, answers = questions.to(device), answers.to(device)

            # Shift answers for teacher forcing
            inputs = answers[:, :-1]
            targets = answers[:, 1:]

            outputs = model(questions, inputs)
            loss = criterion(outputs.reshape(-1, outputs.size(-1)), targets.reshape(-1))
            total_loss += loss.item()

    print(f"Validation Loss: {total_loss / len(val_loader)}")

In [ ]:
# Define the learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define scheduler and train the model
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

train_model(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    num_epochs=100
)



100%|██████████| 12/12 [00:11<00:00,  1.06it/s]


Epoch 1, Loss: 6.3520132303237915


100%|██████████| 12/12 [00:08<00:00,  1.39it/s]


Epoch 2, Loss: 6.092429439226787


100%|██████████| 12/12 [00:07<00:00,  1.62it/s]


...

Epoch 98, Loss: 0.11166313166419665


100%|██████████| 12/12 [00:08<00:00,  1.38it/s]


Epoch 99, Loss: 0.10437614036103089


100%|██████████| 12/12 [00:07<00:00,  1.61it/s]

Epoch 100, Loss: 0.10692537824312846


In [ ]:

# Save only the model state
torch.save(model.state_dict(), "chatbot_model_state.pth")

# To load the model state
loaded_model = Transformer(
    src_vocab_size=question_tokenizer.vocab_size,
    tgt_vocab_size=answer_tokenizer.vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    dropout=dropout
)
loaded_model.load_state_dict(torch.load("/content/chatbot_model_state.pth"))
loaded_model.eval()  # Set the model to evaluation mode



<ipython-input-9-c850ca390dfb>:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model.load_state_dict(torch.load("/content/chatbot_model_state.pth"))


Transformer(
  (src_embedding): Embedding(812, 128)
  (tgt_embedding): Embedding(652, 128)
  (positional_encoding): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-11): 12 x EncoderLayer(
      (self_attn): MultiHeadAttention(
        (query): Linear(in_features=128, out_features=128, bias=True)
        (key): Linear(in_features=128, out_features=128, bias=True)
        (value): Linear(in_features=128, out_features=128, bias=True)
        (out): Linear(in_features=128, out_features=128, bias=True)
      )
      (ffn): PositionwiseFeedForward(
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (decoder_layers): ModuleList(
    (0-11):

In [ ]:
def generate_response(model, question, question_tokenizer, answer_tokenizer, max_len=50):
    model.eval()
    with torch.no_grad():
        # Tokenize and pad the input question
        question_seq = question_tokenizer.texts_to_sequences([question])
        question_seq = pad_sequences(question_seq, max_len)
        question_tensor = torch.tensor(question_seq, dtype=torch.long).to(device)

        # Start with the <SOS> token
        generated_seq = torch.tensor([[1]], dtype=torch.long).to(device)  # <SOS> token

        for _ in range(max_len):
            # Forward pass through the model
            output = model(question_tensor, generated_seq)
            output = output[:, -1, :]  # Get the last token prediction
            try:
                split_size = output.size(-1) // 2  # Example: split into two equal parts
                chunks = torch.split(output, split_size, dim=-1)
                # Assuming some post-processing on chunks is needed
                output = torch.cat(chunks, dim=-1)  # Recombine (if applicable)
            except Exception as e:
                print(f"Split error: {e}")

            next_token = torch.argmax(output, dim=-1).item()

            # Append predicted token
            generated_seq = torch.cat([generated_seq, torch.tensor([[next_token]], dtype=torch.long).to(device)], dim=1)

            # Stop if <EOS> token is generated
            if next_token == 2:  # <EOS> token
                break

        # Convert generated sequence back to text
        response = answer_tokenizer.sequences_to_texts(generated_seq.cpu().numpy())[0]
        return ' '.join(response)
example_question = "Hi , What can yog do ?"
response = generate_response(loaded_model, example_question, question_tokenizer, answer_tokenizer)
print(f"Question: {example_question}")
print(f"Response: {response}")



Question: Hi , What can yog do ?
Response: <SOS> I can help with a variety of things, from answering questions to assisting with tasks. What would you like help with? <EOS>


In [ ]:
example_question = "Hey ,@ make you work 24/7 ?"
response = generate_response(loaded_model, example_question, question_tokenizer, answer_tokenizer)
print(f"Question: {example_question}")
print(f"Response: {response}")

Question: Hey ,@ make you work 24/7 ?
Response: <SOS> Yes, I am available 24/7 to help you! what can I assist you with? <EOS>


In [ ]:
!pip install nltk scikit-learn rouge-score

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

import torch

# Example chatbot responses and references
responses = ["Please report the bug using the Report a Bug feature including as much detail as possible about the issue you encountered"]
references = ["Please report the bug using the Report a Bug feature including as much detail as possible about the issue you encountered"]

# 1. BLEU Score Calculation
def calculate_bleu(responses, references):
    bleu_scores = []
    smoothie = SmoothingFunction().method4
    for response, reference in zip(responses, references):
        score = sentence_bleu(reference, response.split(), smoothing_function=smoothie)
        bleu_scores.append(score)
    return bleu_scores

# 2. ROUGE Score Calculation
def calculate_rouge(responses, references):
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = []
    for response, reference in zip(responses, references):
        scores = rouge.score(response, reference[0])
        rouge_scores.append(scores)
    return rouge_scores

def calculate_meteor(responses, references):
    meteor_scores = []
    for response, reference in zip(responses, references):
        # Ensure reference is a list of tokenized sentences (even if only one)
        reference_tokenized = [reference.split()]  # Wrap in a list

        # Tokenize the response
        response_tokenized = response.split()

        score = meteor_score(reference_tokenized, response_tokenized)
        meteor_scores.append(score)
    return meteor_scores


# Evaluate metrics
bleu_scores = calculate_bleu(responses, references)
rouge_scores = calculate_rouge(responses, references)
meteor_scores = calculate_meteor(responses, references)

# Print results
print("BLEU Scores:", bleu_scores)
print("ROUGE Scores:", rouge_scores)
print("METEOR Scores:", meteor_scores)

BLEU Scores: [0.012518377344512808]
ROUGE Scores: [{'rouge1': Score(precision=0.0, recall=0.0, fmeasure=0.0), 'rouge2': Score(precision=0.0, recall=0.0, fmeasure=0.0), 'rougeL': Score(precision=0.0, recall=0.0, fmeasure=0.0)}]
METEOR Scores: [0.9999460101500918]
